In [1]:
import os
import sys
sys.path.append('../')
sys.path.append("../../")
import numpy as np
import matplotlib.pyplot as plt
import os
from pydicom import dcmread
import torch

from types import MappingProxyType

from pydose_rt import ModelConfig
from pydose_rt.engine.data import DataGenerator
import pydose_rt.utils.plot_utils as plot_utils
from pydose_rt.layers import *


from pydose_rt.DoseEngine import DoseEngine

/home/rd/anaconda3/envs/autoplan/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-30 14:08:02.399364: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
config = ModelConfig(ct_array_shape=(64,64,160), resolution=(0.3, 0.3, 0.3), field_size=(40, 40), number_of_leaf_pairs=5, tpr_20_10=0.72, number_of_cps=1, mu_scaling=1.8, starting_angle=0)
print(config)
x_ct = 0.0 * np.expand_dims(np.ones(config.ct_array_shape), 0)
y_mlc = np.zeros((1, 2, config.number_of_cps, config.number_of_leaf_pairs))
y_mlc[:, 0, :, :] = 0.5
y_mlc[:, 1, :, :] = 1.0
mus = np.ones((1, config.number_of_cps), dtype=np.float32)

ct_array_shape=(64, 64, 160) resolution=(0.3, 0.3, 0.3) field_size=(40, 40) downsampling_factor=(1, 1, 1) iso_center=(0.0, 0.0, 0.0) minimum_leaf_overlap=0.5 maximum_leaf_speed=2.25 minimum_gantry_angle_speed=0.1 maximum_gantry_angle_speed=6.0 maximum_gantry_angle_speed_variation=0.75 minimum_dose_rate=50.0 maximum_dose_rate=600.0 is_fff=True mu_scaling=1.8 focal_spot_sigma=0.15 focus_to_collimator=49.7 oar_coeffs=(1.0, -0.0001, 2.5e-07) mlc_thickness=6.8 mlc_mu=0.7 dtype=torch.float32 device=device(type='cuda') number_of_leaf_pairs=5 tpr_20_10=0.72 mean_photon_energy_MeV=10.0 SID=100.0 number_of_cps=1 starting_angle=0.0 mlc_transmission=0.00856560939749806 gantry_angles=array([0.]) depth_offset=100.0 gantry_diff=6.283185307179586 gantry_diff_deg=360.0 iso_center_in_pixels=array([0, 0, 0], dtype=int32) field_size_in_pixels=(134, 134) leaf_size=8.0 leaf_widths=array([8., 8., 8., 8., 8.], dtype=float32) fluence_profile=(array([ 0.  ,  1.  ,  2.  ,  3.  ,  4.  ,  5.  ,  7.5 , 10.  , 12.5 

In [3]:
notebook_dir = os.getcwd()
parent_dir = "/home/rd/Documents/github/autoplan"
data_path = os.path.join(parent_dir, "database/AUTORPT/")
gen = DataGenerator(data_path, "plotting", True, 1)

Number of files: 1 in plotting cohort


In [4]:
from pydose_rt.pydose_rt.utils.test_utils import TestSetup

In [5]:
def prepare_real(is_hu=False):
    T = TestSetup(
        parent_dir=parent_dir,
        # filedir="database/testsetup/pytorch_b003_bs01_f10_v16_d04_ptv100_fixed_DoseEngine/data_00_c1.npz",
    )
    T.create_dummy(number_of_leaf_pairs=128, number_of_cps=15)

    ct = T.ct  # numpy array
    ct_np = ct
    ptv = T.data["masks"][0, ..., 0]

    # ct_np = downsample_ct_by_2(ct_np)  # Assumes it returns a numpy array
    # ptv = downsample_ct_by_2(ptv)  # Assumes it returns a numpy array
    X, Y, Z = ct_np.shape
    ct_torch = torch.tensor(ct_np, dtype=torch.float32)

    if is_hu:
        # Convert normalized CT back to HU
        HU_MIN = -1000
        HU_MAX = 3000
        ct_torch = ((ct_torch + 1) / 2) * (HU_MAX - HU_MIN) + HU_MIN
    return ct_torch, ptv, X, Y, Z

In [7]:
batch_size = 1
number_of_cps = 3
num_leafs = 60
device = "cuda" if torch.cuda.is_available() else "cpu"

ct_data, ptv, W, D, H = prepare_real(is_hu=False)

T = TestSetup(
    parent_dir=parent_dir,
)
T.create_real()

config = ModelConfig(
    ct_array_shape=(W, D, H),
    number_of_leaf_pairs=num_leafs,
    number_of_cps=number_of_cps,
    field_size=(num_leafs * 1.0, num_leafs * 1.0),
    tpr_20_10=0.72,
)

ct_data = ct_data.unsqueeze(0).expand(batch_size, -1, -1, -1)

x_ct = np.repeat(ct_data, 2, 0)

plt.imshow(x_ct[0, :, :, 80], cmap="gray")

Number of files: 1 in plotting cohort


TypeError: 'module' object is not callable

In [ ]:
print(x_ct.shape)
print(x_ct.max())
print(x_ct.min())

In [ ]:
def compute_plot(x_ct, y_mlc, mus, dose_layer, epoch):
    dose = dose_layer(x_ct, y_mlc, mus)
    print(dose.shape)

    # by z
    slice_idx = x_ct.shape[3] // 2
    plt.imshow(x_ct[0, :, :, slice_idx].cpu(), cmap="gray")
    plt.imshow(dose[0, :, :, slice_idx].cpu(), cmap="jet", alpha=0.2)
    plt.colorbar()
    plt.show()

    # y
    slice_idx = x_ct.shape[2] // 2
    plt.imshow(x_ct[0, :, slice_idx, :].cpu(), cmap="gray")
    plt.imshow(dose[0, :, slice_idx, :].cpu(), cmap="jet", alpha=0.2)
    plt.colorbar()
    plt.show()
    
    # x
    slice_idx = x_ct.shape[1] // 2
    plt.imshow(x_ct[0, slice_idx, :, :].cpu(), cmap="gray")
    plt.imshow(dose[0, slice_idx, :, :].cpu(), cmap="jet", alpha=0.2)
    plt.colorbar()
    plt.show()    

    return dose


def process(config):
    print()
    print()
    # print("config:", config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    fluence_map_layer = FluenceMapLayer(
        config
    )  # Instantiate the FluenceMapLayer
    fluence_volume_layer = FluenceVolumeLayer(
        config
    )  # Instantiate the FluenceVolumeLayer with config

    y_mlc = np.zeros((2, 2, config.number_of_cps, config.number_of_leaf_pairs))
    
    
    center_leaf_start_idx = (60 // 2) - (10 // 2)
    center_leaf_end_idx = center_leaf_start_idx + 10
    y_mlc[:, 0, :, center_leaf_start_idx:center_leaf_end_idx] = 0.5
    y_mlc[:, 1, :, center_leaf_start_idx:center_leaf_end_idx] = 0.5

    y_mlc[:, 0, :, :] = 0.5
    y_mlc[:, 1, :, :] = 0.1
    
    # y_mlc[:, 0, :, :] = 0.5
    # y_mlc[:, 1, :, :] = 0.5 

    fluence_map = fluence_map_layer(torch.tensor(y_mlc, device=device)).cpu()
    # plt.imshow(fluence_map[0, 0, :, :].cpu())
    # plt.colorbar()
    # plt.show()

    fluence_volume = fluence_volume_layer(
        torch.tensor(fluence_map, device=device)
    ).cpu()
    print("fluence_volume shape:", np.shape(fluence_volume))

    slice_number = int(np.shape(fluence_volume)[3] / 2)
    slice_y_number = int(np.shape(fluence_volume)[2] / 2)
    # plt.imshow(fluence_volume[0, :, :, slice_number, 0], interpolation="none")
    # plt.colorbar()
    # plt.show()

    dose_layer = AccumulateDose3DLayer(config, 15)
    save_path = os.path.join(parent_dir, "database/temp/")
    # y_mlc = np.zeros((2, 2, config.number_of_cps, config.number_of_leaf_pairs))
    mus = np.array(np.ones((2, config.number_of_cps)), dtype=np.float32) * 1
    mus = mus * 0.12 / config.number_of_cps
    # mus = mus * (1.5 / 0.464)
    
    dose = compute_plot(
        torch.tensor(x_ct * 1000, device=device, dtype=torch.float32),
        torch.tensor(y_mlc, device=device, dtype=torch.float32),
        torch.tensor(mus, device=device, dtype=torch.float32),
        dose_layer,
        epoch=0,
    ).cpu()
    print("dose min and max:", dose.numpy().min(), dose.numpy().max())
    print("dose sum:", np.sum(dose.numpy()))
    print()
    print()

In [ ]:
process(
    config = ModelConfig(
        ct_array_shape=(W, D, H),
        resolution=(0.3, 0.3, 0.3),
        # resolution=(1,1,1),
        field_size=(40, 40),
        number_of_leaf_pairs=60,
        tpr_20_10=0.72,
        number_of_cps=1,
    )
)
process(
    config = ModelConfig(
        ct_array_shape=(W, D, H),
        resolution=(0.3, 0.3, 0.3),
        # resolution=(1,1,1),
        field_size=(40, 40),
        number_of_leaf_pairs=60,
        tpr_20_10=0.72,
        number_of_cps=3,
    )
)
process(
    config = ModelConfig(
        ct_array_shape=(W, D, H),
        resolution=(0.3, 0.3, 0.3),
        # resolution=(1,1,1),
        field_size=(40, 40),
        number_of_leaf_pairs=60,
        tpr_20_10=0.72,
        number_of_cps=15,
    )
)
# process(
#     config = ModelConfig(
#         ct_array_shape=(128, 128, 320),
#         resolution=(0.3, 0.3, 0.3),
#         field_size=(40, 40),
#         number_of_leaf_pairs=60,
#         tpr_20_10=0.72,
#         number_of_cps=180,
#     )
# )